In [1]:
# If needed (run once):
!pip install -q sentence-transformers scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
REPO_ROOT = Path("..").resolve()
DATA_PROCESSED = REPO_ROOT / "data" / "processed"

jobs_clean = pd.read_parquet(DATA_PROCESSED / "jobs_clean.parquet")
resumes_clean = pd.read_parquet(DATA_PROCESSED / "resumes_clean.parquet")

print("jobs_clean:", jobs_clean.shape)
print("resumes_clean:", resumes_clean.shape)

display(jobs_clean.head(2))
display(resumes_clean.head(2))


jobs_clean: (1068, 6)
resumes_clean: (1200, 7)


,job_id,job_title,experience_level,years_of_experience,job_skills_list,job_text
0,NET-F-001,.NET Developer,fresher,0-1,"[c#, vb.net, .net framework, .net core, asp.ne...",Job Title: .NET Developer. Experience Level: f...
1,NET-F-002,.NET Developer,fresher,0-1,"[c#, .net framework, asp.net, razor, html, css...",Job Title: .NET Developer. Experience Level: f...


,resume_id,current_job_title,education_blob,experience_years,resume_skills_list,target_job_description,resume_text
0,R_0000,,Master's | Master's in Cybersecurity | Cyberse...,0,"[node.js, javascript, deep learning, statistic...",Seeking a challenging role as a Software Devel...,Current Title: . Previous Titles: . Education:...
1,R_0001,Cybersecurity Engineer,Bachelor's | Bachelor's in Electronics Enginee...,5,"[spark, kubernetes, terraform, natural languag...",Targeting a Cybersecurity Engineer position to...,Current Title: Cybersecurity Engineer. Previou...


In [3]:
import re
import numpy as np

def parse_years_range(x: str):
    """
    Convert strings like:
    - '0-1' -> (0, 1)
    - '4-7' -> (4, 7)
    - '9–12 years' -> (9, 12)
    - '10+ years' -> (10, None)
    - '' -> (None, None)
    """
    if x is None:
        return (None, None)
    s = str(x).lower().strip()
    s = s.replace("years", "").replace("year", "").strip()
    s = s.replace("–", "-")  # en-dash to hyphen

    if not s:
        return (None, None)

    # 10+ or 7+
    m = re.match(r"(\d+)\s*\+", s)
    if m:
        return (int(m.group(1)), None)

    # range like 4-7
    m = re.match(r"(\d+)\s*-\s*(\d+)", s)
    if m:
        return (int(m.group(1)), int(m.group(2)))

    # single number
    m = re.match(r"(\d+)", s)
    if m:
        v = int(m.group(1))
        return (v, v)

    return (None, None)

# Add numeric min/max years for jobs
jobs_clean["min_years"], jobs_clean["max_years"] = zip(*jobs_clean["years_of_experience"].map(parse_years_range))
jobs_clean[["years_of_experience", "min_years", "max_years"]].head(10)


,years_of_experience,min_years,max_years
0,0-1,0,1.0
1,0-1,0,1.0
2,0-1,0,1.0
3,0-1,0,1.0
4,0-1,0,1.0
5,0-1,0,1.0
6,0-1,0,1.0
7,0-1,0,1.0
8,0-1,0,1.0
9,0-1,0,1.0


In [4]:
def experience_penalty(resume_years: int, job_min, job_max):
    """
    Returns a penalty in [0, 1] where 1 = perfect alignment.
    Penalizes cases where resume_years is below job_min.
    """
    if job_min is None:
        return 1.0

    # If job has min requirement and resume below it, penalize
    if resume_years < job_min:
        gap = job_min - resume_years
        # each year gap reduces score a bit; cap at strong penalty
        return max(0.4, 1.0 - 0.12 * gap)

    # If resume meets minimum, no penalty
    return 1.0

def skill_overlap_ratio(resume_skills, job_skills):
    r = set(list(resume_skills))
    j = set(list(job_skills))
    if len(j) == 0:
        return 0.0
    return len(r.intersection(j)) / len(j)


def rerank_jobs_for_resume(resume_idx: int, k: int = 10, alpha: float = 0.75):
    """
    alpha controls how much we trust semantic similarity vs experience alignment.
    final_score = alpha*semantic + (1-alpha)*experience_adjusted_semantic
    """
    # baseline similarity
    sims = cosine_similarity(resume_emb[resume_idx:resume_idx+1], job_emb)[0]

    resume_years = int(resumes_clean.loc[resume_idx, "experience_years"])

    # compute adjusted score with experience penalty
    penalties = np.array([
        experience_penalty(resume_years, mn, mx)
        for mn, mx in zip(jobs_clean["min_years"], jobs_clean["max_years"])
    ])

    adjusted = sims * penalties
    skill_ratios = np.array([
    skill_overlap_ratio(
        resumes_clean.loc[resume_idx, "resume_skills_list"],
        js
        )
        for js in jobs_clean["job_skills_list"]
    ])
    
    final = 0.6 * sims + 0.25 * adjusted + 0.15 * skill_ratios


    top_idx = np.argsort(final)[::-1][:k]
    out = jobs_clean.iloc[top_idx][["job_id", "job_title", "experience_level", "years_of_experience", "job_skills_list"]].copy()
    out["semantic_score"] = sims[top_idx]
    out["final_score"] = final[top_idx]
    out["exp_penalty"] = penalties[top_idx]
    return out.reset_index(drop=True)


In [5]:
from collections import Counter

# Build frequency map across all job skills
job_skill_freq = Counter()
for skills in jobs_clean["job_skills_list"]:
    job_skill_freq.update(skills)

def top_missing_skills(resume_skills, job_skills, top_n=10):
    matched, missing = skill_gap(resume_skills, job_skills)

    # Rank missing skills by global frequency (more demanded skills first)
    missing_sorted = sorted(
        missing,
        key=lambda s: job_skill_freq.get(s, 0),
        reverse=True
    )

    return matched, missing_sorted[:top_n]


In [6]:
# Fast and strong baseline for semantic similarity
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [7]:
# Convert to lists of strings
job_texts = jobs_clean["job_text"].astype(str).tolist()
resume_texts = resumes_clean["resume_text"].astype(str).tolist()

# Encode to embeddings (numpy arrays)
job_emb = model.encode(
    job_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True  # important: makes cosine similarity faster & stable
)

resume_emb = model.encode(
    resume_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("job_emb shape:", job_emb.shape)
print("resume_emb shape:", resume_emb.shape)


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

job_emb shape: (1068, 384)
resume_emb shape: (1200, 384)


In [8]:
np.save(DATA_PROCESSED / "job_emb.npy", job_emb)
np.save(DATA_PROCESSED / "resume_emb.npy", resume_emb)

print("Saved embeddings to data/processed (local only).")


Saved embeddings to data/processed (local only).


In [9]:
def top_k_jobs_for_resume(resume_idx: int, k: int = 10):
    """
    Returns top-k job matches for a given resume index.
    """
    # cosine similarity between one resume and all jobs
    sims = cosine_similarity(resume_emb[resume_idx:resume_idx+1], job_emb)[0]
    top_idx = np.argsort(sims)[::-1][:k]

    results = jobs_clean.iloc[top_idx][["job_id", "job_title", "experience_level", "years_of_experience", "job_skills_list"]].copy()
    results["score"] = sims[top_idx]
    return results.reset_index(drop=True)


def top_k_resumes_for_job(job_idx: int, k: int = 10):
    """
    Returns top-k resume matches for a given job index.
    """
    sims = cosine_similarity(job_emb[job_idx:job_idx+1], resume_emb)[0]
    top_idx = np.argsort(sims)[::-1][:k]

    results = resumes_clean.iloc[top_idx][["resume_id", "current_job_title", "experience_years", "resume_skills_list"]].copy()
    results["score"] = sims[top_idx]
    return results.reset_index(drop=True)


In [10]:
def skill_gap(resume_skills, job_skills):
    """
    Computes matched and missing skills safely.
    Works whether input is list or numpy array.
    """
    # Convert to list safely
    r = set(list(resume_skills) if resume_skills is not None else [])
    j = set(list(job_skills) if job_skills is not None else [])

    matched = sorted(r.intersection(j))
    missing = sorted(j.difference(r))

    return matched, missing


In [11]:
from collections import Counter

# Build frequency map across all job skills
job_skill_freq = Counter()
for skills in jobs_clean["job_skills_list"]:
    job_skill_freq.update(skills)

def top_missing_skills(resume_skills, job_skills, top_n=10):
    matched, missing = skill_gap(resume_skills, job_skills)

    # Rank missing skills by global frequency (more demanded skills first)
    missing_sorted = sorted(
        missing,
        key=lambda s: job_skill_freq.get(s, 0),
        reverse=True
    )

    return matched, missing_sorted[:top_n]


In [12]:
resume_idx = 0
matches2 = rerank_jobs_for_resume(resume_idx, k=10, alpha=0.65)

resume_skills = resumes_clean.loc[resume_idx, "resume_skills_list"]

matches2["matched_skills"] = matches2["job_skills_list"].apply(
    lambda js: top_missing_skills(resume_skills, js, top_n=10)[0]
)

matches2["missing_skills_top10"] = matches2["job_skills_list"].apply(
    lambda js: top_missing_skills(resume_skills, js, top_n=10)[1]
)

display(matches2[[
    "job_id",
    "job_title",
    "final_score",
    "matched_skills",
    "missing_skills_top10"
]].head(5))


,job_id,job_title,final_score,matched_skills,missing_skills_top10
0,CSA-F-009,Cybersecurity Trainee,0.576709,[],"[c++, wireshark, metasploit, encryption, linux..."
1,CSA-F-008,Graduate Security Analyst,0.576322,[],"[python, linux, snort, windows, nist, antivirus]"
2,CSA-F-003,Cybersecurity Intern,0.571907,[],"[python, linux, vulnerability assessment, snor..."
3,CA004,Cloud Trainee,0.571062,[],"[bash, python scripting, iam, cloud storage, b..."
4,CSA-F-001,Cybersecurity Analyst,0.562633,[],"[python, c++, linux, wireshark, incident respo..."


In [13]:
job_idx = 0
cand = top_k_resumes_for_job(job_idx, k=5)

print("Job:", jobs_clean.loc[job_idx, "job_id"])
print("Title:", jobs_clean.loc[job_idx, "job_title"])
print("Job skills:", jobs_clean.loc[job_idx, "job_skills_list"])

display(cand)

job_skills = jobs_clean.loc[job_idx, "job_skills_list"]

cand["matched_skills"] = cand["resume_skills_list"].apply(lambda rs: skill_gap(rs, job_skills)[0])
cand["missing_skills"] = cand["resume_skills_list"].apply(lambda rs: skill_gap(rs, job_skills)[1])  # missing from resume vs job

display(cand[["resume_id", "current_job_title", "score", "matched_skills", "missing_skills"]])


Job: NET-F-001
Title: .NET Developer
Job skills: ['c#' 'vb.net' '.net framework' '.net core' 'asp.net' 'mvc' 'html' 'css'
 'javascript' 'sql server' 'entity framework' 'linq' 'visual studio' 'git'
 'unit testing']


,resume_id,current_job_title,experience_years,resume_skills_list,score
0,R_0234,Software Developer,7,"[rest apis, natural language processing, pytho...",0.666126
1,R_0217,,0,"[solidity, microservices, jenkins, penetration...",0.661785
2,R_0527,Software Developer,9,"[python, natural language processing, react, n...",0.632485
3,R_0644,Software Engineer,1,"[javascript, scrum, react, penetration testing...",0.617760
4,R_0485,Software Developer,2,"[java, azure, cybersecurity, scrum, tensorflow...",0.613645


,resume_id,current_job_title,score,matched_skills,missing_skills
0,R_0234,Software Developer,0.666126,[javascript],"[.net core, .net framework, asp.net, c#, css, ..."
1,R_0217,,0.661785,[],"[.net core, .net framework, asp.net, c#, css, ..."
2,R_0527,Software Developer,0.632485,[],"[.net core, .net framework, asp.net, c#, css, ..."
3,R_0644,Software Engineer,0.617760,[javascript],"[.net core, .net framework, asp.net, c#, css, ..."
4,R_0485,Software Developer,0.613645,[],"[.net core, .net framework, asp.net, c#, css, ..."
